In [ ]:
#| default_exp logger

# logger

> simple logger using idiomatic Solveit

In [ ]:
#| export
import asyncio, time
from fastcore.all import patch
from datetime import datetime
from dialoghelper.core import update_msg, add_msg, del_msg, read_msg, find_msg_id

In [ ]:
import random
import fastcore.all as FC
from fastcore.test import *

In [ ]:
#| export
class Logger:
    logs:list; msgid:str; _s:str=''
    def __init__(self, id:str|None=''): self.logs = []; self.setup(id)
    def setup(self, id:str|None='', clear:bool=False): 
        self.msgid = id or (None if id is None else add_msg('' if clear else self._s, msg_type='raw'))
        if id != '':
            if clear: self.clear()
            elif id: update_msg(self.msgid, msg_type='raw')
    def clear(self): 
        self.logs.clear(); self._s=''; 
        if self.msgid: update_msg(self.msgid, content='', msg_type='raw')
    def __call__(self, msg, *args, **kwargs): 
        s = f"[{datetime.now():%H:%M:%S}] {msg}"
        self.logs.insert(0, s)
        if self.msgid:
            self._s = f"\u200b{s}\n" + self._s
            update_msg(self.msgid, content=self._s)
        else: print(s, flush=True)


In [ ]:
@patch
def show(self:Logger, clear:bool=False):
    if not self.msgid: return
    del_msg(self.msgid)
    self.setup('', clear)


In [ ]:
log = Logger()


In [ ]:
log('test')
log('test2')
log('test3')

In [ ]:
log(s := ';qwedcv fjkds')
test_is(s in log.logs[0], True)

In [ ]:
log.logs

['[09:48:09] ;qwedcv fjkds',
 '[09:48:09] test3',
 '[09:48:09] test2',
 '[09:48:09] test']

In [ ]:
log.clear()

In [ ]:
log.logs

[]

In [ ]:
log(f"Some msg {random.randint(0, 1000)}")

In [ ]:
log.logs

['[08:15:11] Some msg 508']

In [ ]:
@patch
def show(self:Logger, clear:bool=False, replace:bool=True):
    if not self.msgid: return
    del_msg(self.msgid)
    self.setup(find_msg_id() if replace else '', clear)
    time.sleep(0.2)
    if replace: update_msg(id=self.msgid, msg_type='raw', content=self._s)

In [ ]:
# log.show() # duplicate, uncomment and run

In [ ]:
#| export
_notfound = {'msg': {}}
def findlogtarget(id:str='', n:int=1):
    if (msg := read_msg(0, id=id) if id else read_msg(n)) == _notfound: return ''
    return msg.id if (cnt := msg.content).count('\n') == cnt.count('\u200b')-1 else ''

In [ ]:
test_eq(findlogtarget(log.msgid), log.msgid)

In [ ]:
test_eq(findlogtarget(), read_msg(1).id)

In [ ]:
test_eq(findlogtarget(n=-1), read_msg().id)

In [ ]:
#| export
@patch
def show(self:Logger, clear:bool=False, replace:bool=False):
    if not self.msgid: return
    del_msg(self.msgid)
    self.setup(find_msg_id() if replace else findlogtarget(), clear)
    time.sleep(0.2)
    if self.msgid: update_msg(id=self.msgid, msg_type='raw', content=self._s)

In [ ]:
add_msg(content='\u200b[09:53:00] some other msg\n\u200b[09:44:16] some msg\n', id=read_msg(1).id, msg_type='raw');

In [ ]:
log.show()

In [ ]:
log.show()

In [ ]:
# log.show(replace=True)  # duplicate, uncomment and run 

Look at `show` method. What it does?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The `show` method relocates the logger's output message in the dialog:

1. Deletes the current log message (at `self.msgid`)
2. Calls `setup()` to create a new message at a different location:
   - If `replace=True`: uses `find_msg_id()` (current message location)
   - Otherwise: uses `findlogtarget()` to find the previous raw message that looks like a log (has `\u200b` markers matching newlines)
3. Waits 0.2 seconds
4. Updates the new message with the log content (`self._s`)
5. If `clear=True` is passed, it also clears the log history

So it essentially moves the log output to either replace the current message or append to an existing log message found above in the dialog.

Very good. Looking at the tools schema of `del_msg`, `add_msg`, `update_msg`, and the method `setup`, is this function blocking in the sense that after a message that calls it has finished running ends up with the correct dialog state, that is , the next message cell is the correct output target?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Looking at the code, `show()` is **not reliably blocking** in that sense.

The issue is that `del_msg()`, `add_msg()`, and `update_msg()` are likely asynchronous operations that update the dialog in the browser. The `time.sleep(0.2)` is a workaround attempt to wait for these operations to complete, but:

1. There's no guarantee 0.2 seconds is enough
2. The sleep happens *after* `setup()` creates the new message, but the final `update_msg()` call happens *after* the sleep - so the dialog state may not be fully settled when `show()` returns
3. `findlogtarget()` and `find_msg_id()` read the dialog state, which might be stale if previous operations haven't completed

So if you call `show()` and immediately run the next cell, the message IDs and positions might not be what you expect. The 0.2s sleep helps but doesn't guarantee correctness.

A proper solution would need these dialog operations to be truly synchronous or return promises/have callbacks to ensure completion.

Ok, I want it to be blocking in that sense (dialog state is correct because has a correct output target immediately following the message issuing the `show` call). I don't want it to be async, though. Or better yet, let's create two versions of the method, one async the other sync.

To ensure "correctness" we have to poll using `findlogtarget` looking for the correct output target (that depends on replace) and sleep of say, 0.1.

Let's create both versions. First, the async version:

In [ ]:
@patch
async def showa(self:Logger, clear:bool=False, replace:bool=False):
    if not self.msgid: return
    oldid = self.msgid
    target_id = find_msg_id() if replace else findlogtarget()
    self.setup(target_id, clear)
    if oldid != self.msgid: del_msg(oldid)
    n, t0 = -1 if replace else 1, time.time()
    while findlogtarget(n) != self.msgid:
        if time.time() - t0 > timeout: return
        await asyncio.sleep(0.1)
    if self.msgid: update_msg(id=self.msgid, msg_type='raw', content=self._s)

In [ ]:
await log.showa()

AttributeError: 'Logger' object has no attribute 'showa'

In [ ]:
@patch
def show(self:Logger, clear:bool=False, replace:bool=False):
    if not self.msgid: return
    del_msg(self.msgid)
    target_id = find_msg_id() if replace else findlogtarget()
    self.setup(target_id, clear)
    while findlogtarget(n=-1 if replace else 1) != self.msgid: time.sleep(0.1)
    if self.msgid: update_msg(id=self.msgid, msg_type='raw', content=self._s)

That won't work because if we block the kernel, Solveit won't be able to work properly, and time.sleep() blocks the kernel. We should call asyncio somehow (internal function?) or make the sleeping in another thread (can use fastcore.parallel helpers)

You're right - blocking the kernel with `time.sleep()` will prevent solveit from updating the dialog. Let's use `fastcore.parallel` to run the polling in a separate thread:

In [ ]:
@patch
def show(self:Logger, clear:bool=False, replace:bool=False):
    if not self.msgid: return
    del_msg(self.msgid)
    target_id = find_msg_id() if replace else findlogtarget()
    self.setup(target_id, clear)
    def _wait():
        while findlogtarget(n=-1 if replace else 1) != self.msgid: time.sleep(0.1)
    FC.threaded(_wait)()
    if self.msgid: update_msg(id=self.msgid, msg_type='raw', content=self._s)

But actually, this won't block either - `threaded` returns immediately. If we need it to block the *function* but not the kernel, we need to join the thread:

In [ ]:
@patch
def show(self:Logger, clear:bool=False, replace:bool=False):
    if not self.msgid: return
    del_msg(self.msgid)
    target_id = find_msg_id() if replace else findlogtarget()
    self.setup(target_id, clear)
    def _wait():
        while findlogtarget(n=-1 if replace else 1) != self.msgid: time.sleep(0.1)
    t = FC.threaded(_wait)()
    t.join()
    if self.msgid: update_msg(id=self.msgid, msg_type='raw', content=self._s)

It'd be good to have a (fairly small) timeout to avoid running indefinitely. Logging is important, but not critical at all, so it's ok if it fails (silently)

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->



<details class='tool-usage-details'>
<summary>add_msg(content='@patch\ndef show(self:Logger, clear:bool=False, rep…', msg_type='code')</summary>

```json
{
  "id": "toolu_01HmUzojjuULxNxsjNbadH5x",
  "call": {
    "function": "add_msg",
    "arguments": {
      "content": "@patch\ndef show(self:Logger, clear:bool=False, replace:bool=False, timeout:float=2.0):\n    if not self.msgid: return\n    del_msg(self.msgid)\n    target_id = find_msg_id() if replace else findlogtarget()\n    self.setup(target_id, clear)\n    def _wait():\n        t0 = time.time()\n        while findlogtarget(n=-1 if replace else 1) != self.msgid:\n            if time.time() - t0 > timeout: return\n            time.sleep(0.1)\n    t = FC.threaded(_wait)()\n    t.join()\n    if self.msgid: update_msg(id=self.msgid, msg_type='raw', content=self._s)",
      "msg_type": "code"
    }
  },
  "result": "'_495e1389'"
}
```

</details>

Added a `timeout` parameter (default 2 seconds) that silently returns if the target message isn't found in time.

In [ ]:
def wait(pred, timeout:float=2.0):
    "Wait until `pred` is True or `timeout` is reached without blocking the kernel"
    res = False
    def _wait():
        nonlocal res
        while not pred(): time.sleep(0.1)
        res = True
    t = FC.threaded(_wait)(); t.join(timeout)
    return res

Check the function `wait` and create some examples with edge cases and exceptions.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->



<details class='tool-usage-details'>
<summary>add_msg(content='test_eq(wait(lambda: True), True)', msg_type='code')</summary>

```json
{
  "id": "toolu_015SLhGmm65owegoDUNQs1Vq",
  "call": {
    "function": "add_msg",
    "arguments": {
      "content": "test_eq(wait(lambda: True), True)",
      "msg_type": "code"
    }
  },
  "result": "'_b67d63f6'"
}
```

</details>



<details class='tool-usage-details'>
<summary>add_msg(content='test_eq(wait(lambda: False, timeout=0.3), False)', msg_type='code')</summary>

```json
{
  "id": "toolu_01Eb5JKERnsq9Eb7QXt8j6qz",
  "call": {
    "function": "add_msg",
    "arguments": {
      "content": "test_eq(wait(lambda: False, timeout=0.3), False)",
      "msg_type": "code"
    }
  },
  "result": "'_c125e6af'"
}
```

</details>



<details class='tool-usage-details'>
<summary>add_msg(content='x = 0\ndef delayed(): \n    global x; x += 1; return…', msg_type='code')</summary>

```json
{
  "id": "toolu_01PAjUy5uR2MT8QkMEB2ktqr",
  "call": {
    "function": "add_msg",
    "arguments": {
      "content": "x = 0\ndef delayed(): \n    global x; x += 1; return x > 2\ntest_eq(wait(delayed, timeout=0.5), True)",
      "msg_type": "code"
    }
  },
  "result": "'_a41ca232'"
}
```

</details>



<details class='tool-usage-details'>
<summary>add_msg(content='y = []\ndef sideeffect(): y.append(1); return len(y…', msg_type='code')</summary>

```json
{
  "id": "toolu_01Bbx2M6Kaqq2RVWo8uQ5ZKZ",
  "call": {
    "function": "add_msg",
    "arguments": {
      "content": "y = []\ndef sideeffect(): y.append(1); return len(y) > 5\nwait(sideeffect, timeout=0.3)\ntest_is(len(y) > 2, True)",
      "msg_type": "code"
    }
  },
  "result": "'_409d0282'"
}
```

</details>



<details class='tool-usage-details'>
<summary>add_msg(content='test_eq(wait(lambda: 1/0, timeout=0.2), False)', msg_type='code')</summary>

```json
{
  "id": "toolu_012eFVfiWBW75L6mZw86xBR9",
  "call": {
    "function": "add_msg",
    "arguments": {
      "content": "test_eq(wait(lambda: 1/0, timeout=0.2), False)",
      "msg_type": "code"
    }
  },
  "result": "'_ba96c903'"
}
```

</details>

Created test cases covering: immediate True, timeout on False, delayed success, side effects during polling, and exception in predicate (which will cause the thread to fail but timeout will still work).

In [ ]:
test_eq(wait(lambda: True), True)

In [ ]:
test_eq(wait(lambda: False, timeout=0.3), False)

In [ ]:
x = 0
def delayed(): 
    global x; x += 1; return x > 2
test_eq(wait(delayed, timeout=0.5), True)
test_is(x>2, True)

In [ ]:
y = []
def sideeffect(): y.append(1); return len(y) > 5
wait(sideeffect, timeout=0.3)
test_is(len(y) > 2, True)

In [ ]:
y = []

['_2756fed3', '_74ededbd', '_0b2b0cef']


In [ ]:
int(msg_idx('_2756fed3'))

TypeError: int() argument must be a string, a bytes-like object or a real number, not 'AttrDict'

In [ ]:
if y:
    for msgid in y: del_msg(msgid)
def sideeffect(): y.append(add_msg(str(y))); return len(y) > 2
wait(sideeffect, timeout=2)
test_is(len(y) > 2, True)
print(y)

['_2756fed3']

['_2756fed3', '_74ededbd']

In [ ]:
test_eq(wait(lambda: 1/0, timeout=0.2), False)

Exception in thread Thread-12 (g):
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/local/lib/python3.12/site-packages/ipykernel/ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "/usr/local/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/site-packages/fastcore/parallel.py", line 29, in g
    res = f(*args, **kwargs)
          ^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_11392/3619117207.py", line 6, in _wait
  File "/tmp/ipykernel_11392/4270811551.py", line 1, in <lambda>
ZeroDivisionError: division by zero


How can I check manually the mernel is not blocked while wait is ?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

You can check if the kernel is responsive by running a simple expression that should execute immediately:

```python
1+1
```

Or check the current time:

```python
time.time()
```

If these execute right away and return a result, the kernel isn't blocked. If they hang or don't respond, the kernel is blocked.

In [ ]:
@patch
def show(self:Logger, clear:bool=False, replace:bool=False, timeout:float=2.0):
    if not self.msgid: return
    oldid = self.msgid
    self.setup(find_msg_id() if replace else findlogtarget(), clear)
    if oldid != self.msgid: del_msg(oldid)
    def _wait():
        n, t0 = -1 if replace else 1, time.time()
        while findlogtarget(n) != self.msgid:
            if time.time() - t0 > timeout: return
            time.sleep(0.1)
    t = FC.threaded(_wait)(); t.join()
    if self.msgid: update_msg(self.msgid, msg_type='raw', content=self._s)

# export -

In [ ]:
from dutil.flakes import show_flakes
show_flakes()

No warnings to report

In [ ]:
# #|hide
# #|eval: false
# import fastcore.all as FC
# from nbdev import nbdev_export
# if FC.IN_NOTEBOOK:
#     nb_path = '01_logger.ipynb'
#     nbdev_export(nb_path)